# Exemples cours introduction explicabilité machine learning
ce notebook permet d'illustrer le cours avec des exemples. 
Dans ce cadre, les données et le modèle sont travaillés très rapidement.
<br />

Les data source proviennent de [ERA5](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels?tab=download)






In [ ]:
#Import des librairies
import pandas as pd
import numpy as np
import shap
from shapash.explainer.smart_explainer import SmartExplainer
from sklearn.model_selection import train_test_split
import catboost
import lightgbm
from sklearn.inspection import PartialDependenceDisplay

import warnings
warnings.filterwarnings("ignore")

import plotly
plotly.io.renderers.default = 'png'


In [ ]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', 1000)

## Import du dataset

In [ ]:
df = pd.read_csv("../../data/era5_sample.csv")

In [ ]:
df['valid_time'] = pd.to_datetime(df['valid_time'])
df['hour'] = df['valid_time'].dt.hour
df['wind_speed'] = np.sqrt(df['u10']**2 + df['v10']**2) * 3.6 #en km/h
df['t2m'] = df['t2m'] - 273.15 #en celsius

In [ ]:
df.head(1)

## Modélisation

In [ ]:
features = ['latitude', 'longitude', 'd2m', 't2m', 'ssrd', 'sp', 'tp', 'hour', 'wind_speed']

In [ ]:
df_reduit = df[features].copy()
df_reduit.dropna(inplace=True)

In [ ]:
df_reduit["target"]=df_reduit["wind_speed"]
y = df_reduit["target"]
X = df_reduit.drop(["target", "wind_speed"], axis=1)

In [ ]:
Xtrain, Xtest, ytrain, ytest = train_test_split(
    X, y, train_size=0.75, random_state=1
)

model = catboost.CatBoostRegressor()
model.fit(Xtrain, ytrain, verbose=False)

## Explicabilité

In [ ]:
feature_dict = {
    'd2m': 'Température point de rosée',  
    't2m': 'Température',  
    'ssrd': 'Rayonnement solaire ',
    'sp': 'Pression',  
    'tp': 'Précipitations',  
}

In [ ]:
#Init Shapash
xpl = SmartExplainer(model=model, features_dict=feature_dict)

In [ ]:
# Compilation de Shapash
xpl.compile(
    x=Xtest,    
    y_target = ytest,
)

In [ ]:
#Lancement de la webapp
app=xpl.run_app(port=8080)

In [ ]:
app.kill()

In [ ]:
# PDP par sklearn
features_info = {
    # features of interest
    "features": ["t2m"]}

display = PartialDependenceDisplay.from_estimator(
    model,
    Xtest,
    **features_info,    
)

In [ ]:
xpl.plot.contribution_plot(col="t2m")

In [ ]:
xpl.plot.features_importance()

In [ ]:
xpl.plot.local_plot(index=43660, plot_type='waterfall', waterfall_xaxis_start='auto')

In [ ]:
xpl.plot.local_plot(index=43660)

In [ ]:
xpl.plot.local_plot(index=14317, plot_type='waterfall', waterfall_xaxis_start='auto')

In [ ]:
xpl.plot.local_plot(index=14317)

In [ ]:
xpl.plot.features_importance(mode="global-local")

In [ ]:
model_lgb = lightgbm.LGBMRegressor()
model_lgb.fit(Xtrain, ytrain)

In [ ]:
xpl_lgb = SmartExplainer(model=model_lgb)

xpl_lgb.compile(
    x=Xtest,    
    y_target = ytest
)

In [ ]:
xpl.plot.top_interactions_plot(nb_top_interactions=2)